# Mini-projet Trey 
Maxime Magnenat & Clément Grodent


## 0. Imports & helpers


In [312]:
import numpy as np
import pandas as pd
import pandapower as pp
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pickle
import pp_heig_simulation as pp_sim
import pp_heig_plot as pp_plot
from datetime import time
import re
import os
import copy

In [313]:
# Helpers for time series cleanup (ChatGPT)
def reset_timeseries(net, drop_sgen=False):
    if drop_sgen and len(net.sgen):
        net.sgen.drop(net.sgen.index, inplace=True)
    if hasattr(net, 'controller') and len(net.controller):
        net.controller.drop(net.controller.index, inplace=True)
    if hasattr(net, 'output_writer') and len(net.output_writer):
        net.output_writer.drop(net.output_writer.index, inplace=True)

def extract_p_profiles(file_path, sheet_name, label_prefix):
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0, 1])
    df = df.dropna(how="all", axis=1).dropna(how="all", axis=0)
    df = df.loc[:, df.columns.get_level_values(0).astype(str).str.strip() != ""]
    time_index = df.iloc[:, 0]
    p_cols = [col for col in df.columns if col[1] == "P [MW]"]
    p_df = pd.DataFrame(index=time_index)
    for col in p_cols:
        p_df[f"{label_prefix}_{col[0]}"] = df[col].values * 1000
    return p_df

def _select_profile_column(df, key):
    if key in df.columns:
        return df[[key]]
    if str(key) in df.columns:
        return df[[str(key)]]
    for col in df.columns:
        if str(col) == str(key):
            return df[[col]]
    raise KeyError(f"Profile column {key} not found in profiles")

def build_pv_profile_for_bus(sgen_profiles, bus_id, peak_mw, cosphi=1.0, curtail=1.0, q_sign=1):
    pv_p = _select_profile_column(sgen_profiles["p_mw"], bus_id).copy()
    max_p = pv_p.max().max()
    scale = (peak_mw / max_p) if max_p and max_p > 0 else 0.0
    pv_p = pv_p * scale * curtail
    if cosphi is None or cosphi >= 0.9999:
        pv_q = pv_p * 0.0
    else:
        pv_q = pv_p * np.tan(np.arccos(cosphi)) * q_sign
    pv_p.columns = [bus_id]
    pv_q.columns = [bus_id]
    return pv_p, pv_q, scale

def profile_energy_mwh(p_df):
    if p_df.empty:
        return 0.0
    idx = pd.to_datetime(p_df.index.astype(str))
    step = idx.to_series().diff().dropna().median()
    step_h = step.total_seconds()/3600 if step is not pd.NaT and step is not None else 1.0
    return p_df.sum().sum() * step_h

def summarize_timeseries(result_df):
    v = result_df["res_bus.vm_pu"]
    l = result_df.get("res_line.loading_percent")
    summary = {
        "vmin": v.min().min(),
        "vmax": v.max().max(),
    }
    if l is not None:
        summary["line_loading_max"] = l.max().max()
    return summary



## 1. Import from pickle


In [314]:
## Import pickle
net_trey = pp.from_pickle("input-data/trey_net_student.p")

## 2. Lines parameters


In [315]:
### Line parameters
## Add a new column "length_km" to net_trey
net_trey.line["length_km"] = [0.180, 0.100, 0.115, 0.080, 0.090, 0.135, 0.100, 0.205, 0.085, 0.255, 0.340]

## Create new column "cable_type" by removing the unwanted value (_x) used to differentiate each line from "name" with regex
net_trey.line["cable_type"] = net_trey.line.apply(
    lambda x: re.sub(r"_[0-9]$", "", x["name"]), axis=1
)

## Create parameter table for cable types (datasheets)
cable_types = pd.DataFrame(
    {
        "cable_type": ["GKN3x150_150", "GKN3X95_95", "GKN3X50_50", "GKT3X50_50", "GKN3X240_240"],
        "r_ohm_per_km": [0.124, 0.193, 0.387, 0.387, 0.0754],
        "x_ohm_per_km": [0.07, 0.07, 0.07, 0.07, 0.07],
        "c_nf_per_km": [349, 338, 298, 298, 346],
        "max_i_ka": [0.400, 0.252, 0.170, 0.170, 0.512],
    }
)

## Merge WITHOUT destroying pandapower mandatory columns
tmp = net_trey.line[["cable_type"]].merge(cable_types, on="cable_type", how="left")
for col in ["r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka"]:
    net_trey.line[col] = tmp[col].values

## Mandatory columns for pandapower
net_trey.line["from_bus"] = net_trey.line["from_bus"].astype(int)
net_trey.line["to_bus"] = net_trey.line["to_bus"].astype(int)
net_trey.line["parallel"] = 1

net_trey.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service,length_km,cable_type,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,max_i_ka,parallel
0,GKN3x150_150,0,1,0.0,1.0,None,True,0.180,GKN3x150_150,0.1240,0.07,349,0.400,1
1,GKN3x150_150_2,1,2,0.0,1.0,None,True,0.100,GKN3x150_150,0.1240,0.07,349,0.400,1
2,GKN3X95_95,1,3,0.0,1.0,None,True,0.115,GKN3X95_95,0.1930,0.07,338,0.252,1
3,GKT3X50_50,3,4,0.0,1.0,None,True,0.080,GKT3X50_50,0.3870,0.07,298,0.170,1
4,GKT3X50_50_2,4,5,0.0,1.0,None,True,0.090,GKT3X50_50,0.3870,0.07,298,0.170,1
5,GKN3x150_150_3,0,6,0.0,1.0,None,True,0.135,GKN3x150_150,0.1240,0.07,349,0.400,1
6,GKN3X95_95_2,6,7,0.0,1.0,None,True,0.100,GKN3X95_95,0.1930,0.07,338,0.252,1
7,GKN3x150_150_4,7,8,0.0,1.0,None,True,0.205,GKN3x150_150,0.1240,0.07,349,0.400,1
8,GKN3X50_50,7,9,0.0,1.0,None,True,0.085,GKN3X50_50,0.3870,0.07,298,0.170,1
9,GKN3X50_50_2,6,10,0.0,1.0,None,True,0.255,GKN3X50_50,0.3870,0.07,298,0.170,1


## 3. Bus & external grid


In [317]:
### Bus nominal voltages
net_trey.bus["vn_kv"] = net_trey.bus.apply(
    lambda row: 18.3 if row["type"] == "Slack" else (0.420 if row["type"] == "PQ" else row.get("vn_kv", np.nan)),
    axis=1,
)

# Creat ext_grid 
net_trey.ext_grid.drop(net_trey.ext_grid.index, inplace=True)

slack_bus = int(net_trey.bus.index[net_trey.bus["type"] == "Slack"][0])
pp.create_ext_grid(net_trey, bus=slack_bus, vm_pu=1.0)
net_trey.ext_grid["bus"] = net_trey.ext_grid["bus"].astype(int)


net_trey.bus

,name,type,zone,in_service,vn_kv
0,STMT003438,PQ,Trafo,True,0.42
1,CDBT004764,PQ,North,True,0.42
2,CDBT003746,PQ,North,True,0.42
3,CDBT004760,PQ,North,True,0.42
4,CDBT012139,PQ,North,True,0.42
5,CDBT900784,PQ,North,True,0.42
6,CDBT901452,PQ,South,True,0.42
7,CDBT004774,PQ,South,True,0.42
8,CDBT901604,PQ,South,True,0.42
9,CDBT016055,PQ,South,True,0.42


## 4. loads


In [318]:
### Loads mandatory columns for pandapower / time series
net_trey.load["bus"] = net_trey.load["bus"].astype(int)
# Base load values
net_trey.load["p_mw"] = 0.004
net_trey.load["q_mvar"] = 0.00132
net_trey.load["const_z_percent"] = 20
net_trey.load["const_i_percent"] = 30
net_trey.load["scaling"] = 1.0
net_trey.load["profile_mapping"] = net_trey.load["bus"]

# Scaling per cabinet (allready included in the base load values)
scaling_map = {
    "STMT003438": 1,
    "CDBT004764": 1,
    "CDBT003746": 1,
    "CDBT004760": 1,
    "CDBT012139": 1,
    "CDBT900784": 1,
    "CDBT901452": 1,
    "CDBT004774": 1,
    "CDBT901604": 1,
    "CDBT016055": 1,
    "N1": 1,
    "60437": 1,
}
for name, scale in scaling_map.items():
    net_trey.load.loc[net_trey.load["name"] == name, "scaling"] = scale

net_trey.load


,name,bus,const_z_percent,const_i_percent,sn_mva,in_service,type,p_mw,q_mvar,scaling,profile_mapping
0,STMT003438,0,20,30,None,True,wye,0.004,0.00132,1.0,0
1,CDBT004764,1,20,30,None,True,wye,0.004,0.00132,1.0,1
2,CDBT003746,2,20,30,None,True,wye,0.004,0.00132,1.0,2
3,CDBT004760,3,20,30,None,True,wye,0.004,0.00132,1.0,3
4,CDBT012139,4,20,30,None,True,wye,0.004,0.00132,1.0,4
5,CDBT900784,5,20,30,None,True,wye,0.004,0.00132,1.0,5
6,CDBT901452,6,20,30,None,True,wye,0.004,0.00132,1.0,6
7,CDBT004774,7,20,30,None,True,wye,0.004,0.00132,1.0,7
8,CDBT901604,8,20,30,None,True,wye,0.004,0.00132,1.0,8
9,CDBT016055,9,20,30,None,True,wye,0.004,0.00132,1.0,9


## 5. Power flow to check grid


In [319]:
## Plot grid
pp_plot.plot_power_network(
    net=net_trey,
    plot_title="Trey network",
    filename="trey_grid_example",
)

In [320]:
# Safe fix: set rows with missing bus out of service, then cast bus -> int
for name, df in list(net_trey.items()):
    if isinstance(df, pd.DataFrame) and "bus" in df.columns:
        if "in_service" in df.columns:
            df.loc[df["bus"].isna(), "in_service"] = False
        df = df.dropna(subset=["bus"])
        df["bus"] = df["bus"].astype(int)
        if "in_service" in df.columns:
            df["in_service"] = df["in_service"].astype(bool)
        net_trey[name] = df

pp.runpp(net_trey)
net_trey.res_bus
net_trey.res_ext_grid
pp_plot.plot_powerflow_result(
    net=net_trey,
    plot_title="Power flow results – Trey",
    filename="Result_Power_Flow_Trey_Net",
)


## 6. Timeseries loads only


In [321]:
### Timeseries summer week
reset_timeseries(net_trey, drop_sgen=True)
profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

load_name = os.path.basename(profile_file_path).replace(".xlsx", "")
p_load = extract_p_profiles(profile_file_path, "load", "Load")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads (summer week)",
    filename=os.path.join(load_name + "_power_profiles_all_loads"),
)

# Apply timeseries to net_trey
net_trey.load["profile_mapping"] = net_trey.load["bus"]

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])

result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=19, minute=45)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(load_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage - summer week",
    filename=os.path.join(load_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power - summer week",
    filename=os.path.join(load_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (summer week)",
    filename=os.path.join(load_name + "_line_loading_result"),
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



**Analyse – profils de charge (ete, semaine)**
Le profil ete semaine presente un creux nocturne, une remontee matinale et un pic en soiree. La tension suit l’evolution de la charge: elle baisse aux heures de consommation elevee. Ce scenario sert de reference pour evaluer la marge en tension en ete.


In [322]:
### Timeseries summer weekend
reset_timeseries(net_trey, drop_sgen=True)
profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_weekend.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

load_name = os.path.basename(profile_file_path).replace(".xlsx", "")
p_load = extract_p_profiles(profile_file_path, "load", "Load")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads (summer weekend)",
    filename=os.path.join(load_name + "_power_profiles_all_loads"),
)

# Apply timeseries to net_trey
net_trey.load["profile_mapping"] = net_trey.load["bus"]

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])

result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=12, minute=15)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(load_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage - summer weekend",
    filename=os.path.join(load_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power - summer weekend",
    filename=os.path.join(load_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (summer weekend)",
    filename=os.path.join(load_name + "_line_loading_result"),
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



In [323]:
### Timeseries winter week
reset_timeseries(net_trey, drop_sgen=True)
profile_file_path = "input-data/load_curve/power_profile_cabinet_winter_week.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

load_name = os.path.basename(profile_file_path).replace(".xlsx", "")
p_load = extract_p_profiles(profile_file_path, "load", "Load")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads (winter week)",
    filename=os.path.join(load_name + "_power_profiles_all_loads"),
)

# Apply timeseries to net_trey
net_trey.load["profile_mapping"] = net_trey.load["bus"]

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])

result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=19, minute=45)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(load_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage - winter week",
    filename=os.path.join(load_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power - winter week",
    filename=os.path.join(load_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (winter week)",
    filename=os.path.join(load_name + "_line_loading_result"),
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



In [324]:
### Timeseries winter weekend
reset_timeseries(net_trey, drop_sgen=True)
profile_file_path = "input-data/load_curve/power_profile_cabinet_winter_weekend.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

load_name = os.path.basename(profile_file_path).replace(".xlsx", "")
p_load = extract_p_profiles(profile_file_path, "load", "Load")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads (winter weekend)",
    filename=os.path.join(load_name + "_power_profiles_all_loads"),
)

# Apply timeseries to net_trey
net_trey.load["profile_mapping"] = net_trey.load["bus"]

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])

result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=12, minute=0)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(load_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage - winter weekend",
    filename=os.path.join(load_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power - winter weekend",
    filename=os.path.join(load_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (winter weekend)",
    filename=os.path.join(load_name + "_line_loading_result"),
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



## 7. PV (N1 + 60437)


**Avertissement exécution** : pour les scénarios *PV + charge*, exécuter d’abord les cellules **PV (sgen)** correspondantes, 
afin que les profils sgen soient bien appliqués (sinon le PV n’est pas pris en compte).


In [325]:
### PV (sgen) N1 – summer week
reset_timeseries(net_trey, drop_sgen=True)
pv_profile_path = "input-data/load_curve/power_profile_cabinet_summer_week.xlsx"
time_series_pv = pp_sim.load_power_profile_form_xlsx(file_path=pv_profile_path)

pv_name = os.path.basename(pv_profile_path).replace(".xlsx", "")
p_sgen = extract_p_profiles(pv_profile_path, "sgen", "Sgen")
pp_plot.plot_timeseries_result(
    data_df=p_sgen,
    ylabel="P [kW]",
    plot_title=f"Power profiles of all gen (summer week)",
    filename=os.path.join(pv_name + "_power_profiles_all_gen"),
)

# Create PV generators at bus N1 and 60437
bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])
pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_summer")
pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_summer")

# Map PV profile to the sgen
net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
pp_sim.apply_power_profile(net=net_trey, equipment="sgen", power_profiles=time_series_pv["sgen"])

net_trey.sgen


,name,bus,p_mw,q_mvar,sn_mva,scaling,profile_mapping,type,k,rx,current_source,in_service
0,PV_N1_summer,10,0.0,0.0,NaN,1.0,10,wye,NaN,NaN,True,True
1,PV_60437_summer,11,0.0,0.0,NaN,1.0,11,wye,NaN,NaN,True,True


In [326]:
### PV (sgen) N1 – winter week
reset_timeseries(net_trey, drop_sgen=True)
pv_profile_path = "input-data/load_curve/power_profile_cabinet_winter_week.xlsx"
time_series_pv = pp_sim.load_power_profile_form_xlsx(file_path=pv_profile_path)

pv_name = os.path.basename(pv_profile_path).replace(".xlsx", "")
p_sgen = extract_p_profiles(pv_profile_path, "sgen", "Sgen")
pp_plot.plot_timeseries_result(
    data_df=p_sgen,
    ylabel="P [kW]",
    plot_title=f"Power profiles of all gen (winter week)",
    filename=os.path.join(pv_name + "_power_profiles_all_gen"),
)

# Create PV generators at bus N1 and 60437
bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])
pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_winter")
pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_winter")

# Map PV profile to the sgen
net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
pp_sim.apply_power_profile(net=net_trey, equipment="sgen", power_profiles=time_series_pv["sgen"])

net_trey.sgen


,name,bus,p_mw,q_mvar,sn_mva,scaling,profile_mapping,type,k,rx,current_source,in_service
0,PV_N1_winter,10,0.0,0.0,NaN,1.0,10,wye,NaN,NaN,True,True
1,PV_60437_winter,11,0.0,0.0,NaN,1.0,11,wye,NaN,NaN,True,True


In [327]:
### Loadflow PV + load – summer week
# Load summer load profiles
profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week.xlsx"
pv_run_name = "pv_summer_week"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)
pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

# Ensure PV profiles are applied too (from same file)
if len(net_trey.sgen):
    net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
    pp_sim.apply_power_profile(net=net_trey, equipment="sgen", power_profiles=time_series["sgen"])

# Run timeseries (with PV summer already mapped)
pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])
result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=12, minute=0)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(pv_run_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage – PV summer",
    filename=os.path.join(pv_run_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power – PV summer",
    filename=os.path.join(pv_run_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (summer week)",
    filename=os.path.join(pv_run_name + "_line_loading_result"),
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



In [328]:
### Loadflow PV + load – winter week
# Load winter load profiles
profile_file_path = "input-data/load_curve/power_profile_cabinet_winter_week.xlsx"
pv_run_name = "pv_winter_week"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)
pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

# Ensure PV profiles are applied too (from same file)
if len(net_trey.sgen):
    net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
    pp_sim.apply_power_profile(net=net_trey, equipment="sgen", power_profiles=time_series["sgen"])

# Run timeseries (with PV winter already mapped)
pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw", "res_line.loading_percent"])
result_df = pp_sim.run_time_simulation(net=net_trey)

for plot_time in [time(hour=12, minute=0)]:
    pp_plot.plot_timestamps_powerflow_result(
        net=net_trey,
        plot_time=plot_time,
        filename=os.path.join(pv_run_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage – PV winter",
    filename=os.path.join(pv_run_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power – PV winter",
    filename=os.path.join(pv_run_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading (winter week)",
    filename=os.path.join(pv_run_name + "_line_loading_result"),
)


2026-01-25 20:09:37 Maxime pandapower.io_utils[86310] INFO Updating output_writer with index 0
/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



## 8. PV 200 kW


In [329]:
### PV 200 kW at N1 – summer week (from file)
reset_timeseries(net_trey, drop_sgen=True)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
output_name = "pv_200kw_summer"

# Load profiles
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Keep PV profile for later calculations
pv_p_200 = time_series["sgen"]["p_mw"].copy()

# Load mapping
net_trey.load["profile_mapping"] = net_trey.load["bus"]

# Sgen at N1 and 60437 (garde les autres profils à 0)
bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])
pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")

net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
net_trey.sgen

# Plot profiles (kW)
p_load = extract_p_profiles(profile_file_path, "load", "Load")
p_sgen = extract_p_profiles(profile_file_path, "sgen", "Sgen")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads (summer week)",
    filename=os.path.join(output_name + "_power_profiles_all_loads"),
)
pp_plot.plot_timeseries_result(
    data_df=p_sgen,
    ylabel="P [kW]",
    plot_title=f"Power profiles gen (summer week)",
    filename=os.path.join(output_name + "_power_profiles_all_gen"),
)

# Apply profiles (load + sgen)
for eq in ["load", "sgen"]:
    pp_sim.apply_power_profile(net=net_trey, equipment=eq, power_profiles=time_series[eq])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.loading_percent", "res_line.p_from_mw", "res_bus.vm_pu"])
result_df_200kw = pp_sim.run_time_simulation(net=net_trey)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title=f"Bus voltage ({output_name})",
    filename=os.path.join(output_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title=f"Line power ({output_name})",
    filename=os.path.join(output_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading ({output_name})",
    filename=os.path.join(output_name + "_line_loading_result"),
)

pp_plot.plot_timestamps_powerflow_result(
    net=net_trey,
    plot_time=time(hour=12, minute=0),
    filename=os.path.join(output_name + "_power_flow_at_12h00"),
)

summary_200kw = summarize_timeseries(result_df_200kw)
summary_200kw


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'vmin': 0.9128671636995659,
 'vmax': 1.153993073488985,
 'line_loading_max': 275.71814623481055}

In [330]:
### PV 200 kW at N1 – winter week (from file)
reset_timeseries(net_trey, drop_sgen=True)

profile_file_path = "input-data/load_curve/power_profile_cabinet_winter_week_with_prod_200kW.xlsx"
output_name = "pv_200kw_winter"

# Load profiles
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Load mapping
net_trey.load["profile_mapping"] = net_trey.load["bus"]

# Sgen at N1 and 60437
bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])
pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")

net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]
net_trey.sgen

# Plot profiles (kW)
p_load = extract_p_profiles(profile_file_path, "load", "Load")
p_sgen = extract_p_profiles(profile_file_path, "sgen", "Sgen")
pp_plot.plot_timeseries_result(
    data_df=p_load,
    ylabel="P [kW]",
    plot_title=f"Power profiles loads ({output_name} – winter week)",
    filename=os.path.join(output_name + "_power_profiles_all_loads"),
)
pp_plot.plot_timeseries_result(
    data_df=p_sgen,
    ylabel="P [kW]",
    plot_title=f"Power profiles gen ({output_name} – winter week)",
    filename=os.path.join(output_name + "_power_profiles_all_gen"),
)

# Apply profiles (load + sgen)
for eq in ["load", "sgen"]:
    pp_sim.apply_power_profile(net=net_trey, equipment=eq, power_profiles=time_series[eq])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.loading_percent", "res_line.p_from_mw", "res_bus.vm_pu"])
result_df_200kw_winter = pp_sim.run_time_simulation(net=net_trey)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw_winter["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title=f"Bus voltage ({output_name})",
    filename=os.path.join(output_name + "_voltage_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw_winter["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title=f"Line power ({output_name})",
    filename=os.path.join(output_name + "_line_power_result"),
)

pp_plot.plot_timeseries_result(
    data_df=result_df_200kw_winter["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading ({output_name} – winter week)",
    filename=os.path.join(output_name + "_line_loading_result"),
)

pp_plot.plot_timestamps_powerflow_result(
    net=net_trey,
    plot_time=time(hour=12, minute=0),
    filename=os.path.join(output_name + "_power_flow_at_12h00"),
)

summary_200kw_winter = summarize_timeseries(result_df_200kw_winter)
summary_200kw_winter


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'vmin': 0.9093535573370244,
 'vmax': 1.0170400637406523,
 'line_loading_max': 103.97213283462989}

In [331]:
### PV 200 kW – diagnostics (summer week)
v = result_df_200kw["res_bus.vm_pu"]
l = result_df_200kw["res_line.loading_percent"]

issues = {
    "v_low_buses(<0.90pu)": list(v.columns[(v < 0.90).any()].values),
    "v_high_buses(>1.10pu)": list(v.columns[(v > 1.10).any()].values),
    "overloaded_lines(>100%)": list(l.columns[(l > 100).any()].values),
}
issues


{'v_low_buses(<0.90pu)': [],
 'v_high_buses(>1.10pu)': ['N1'],
 'overloaded_lines(>100%)': ['GKN3x150_150_3', 'GKN3X50_50_2']}

In [ ]:
### Prix électricité (Trey, 2026) – ElCom
price_ct_kwh_trey_2026 = 14.55  # ct/kWh (ElCom Trey 2026)
price_chf_per_mwh = price_ct_kwh_trey_2026 * 10


In [333]:
### PV 200 kW – max injectable estimate (current limit)
# Use current rating before line replacement
I_ka = 0.17  # 170 A
U_ll_kv = 0.4  # 400 V line-to-line
cosphi = 1.0

# Pmax = sqrt(3) * U_ll * I * cosphi
p_max_mw = (np.sqrt(3) * U_ll_kv * I_ka * cosphi)

# Energy & economic impact (scale from 200 kW profile)
profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
if "pv_p_200" not in globals():
    time_series_tmp = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)
    pv_p_200 = time_series_tmp["sgen"]["p_mw"].copy()

energy_base_mwh = profile_energy_mwh(pv_p_200[[10]] if 10 in pv_p_200.columns else pv_p_200)
scale_limit = p_max_mw / 0.2  # relative to 200 kW peak
energy_max_mwh = energy_base_mwh * scale_limit
energy_lost_mwh = energy_base_mwh - energy_max_mwh
loss_chf = energy_lost_mwh * price_chf_per_mwh if price_chf_per_mwh is not None else None

{
    "p_max_mw": p_max_mw,
    "energy_base_mwh": energy_base_mwh,
    "energy_max_mwh": energy_max_mwh,
    "energy_lost_mwh": energy_lost_mwh,
    "loss_chf": loss_chf,
}


/tmp/ipykernel_86310/1488186706.py:47: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



{'p_max_mw': 0.11777945491468367,
 'energy_base_mwh': 3.285770659890386,
 'energy_max_mwh': 1.9349813864827508,
 'energy_lost_mwh': 1.3507892734076354,
 'loss_chf': 386.32573219458374}

**Analyse – puissance injectee maximale**
La limite thermique de la ligne est estimee par Pmax = sqrt(3)*Ull*I*cos(phi). Avec Ull=0.4 kV et I=170 A, on obtient environ 0.118 MW (118 kW). Cette valeur sert de cap pour le bridage avant renforcement de ligne.


In [334]:
### PV 200 kW – curtailment impact (only when PV creates overload)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
output_name = "pv_200kw_summer_curtail"

# Current limit before line change
I_ka = 0.17  # 170 A
U_ll_kv = 0.4  # 400 V
cosphi = 1.0
p_max_mw = (np.sqrt(3) * U_ll_kv * I_ka * cosphi)

# Load profiles once
base_ts = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# --- 1) Load-only baseline (same time index as PV) ---
net_load = copy.deepcopy(net_trey)
reset_timeseries(net_load, drop_sgen=True)

net_load.load["profile_mapping"] = net_load.load["bus"]

bus_n1 = int(net_load.bus.index[net_load.bus["name"] == "N1"][0])
bus_60437 = int(net_load.bus.index[net_load.bus["name"] == "60437"][0])
pp.create_sgen(net_load, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_zero")
pp.create_sgen(net_load, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_zero")
net_load.sgen["profile_mapping"] = net_load.sgen["bus"]

# copy profiles but set PV to zero -> ensures identical time index
base_ts_zero = {
    "load": base_ts["load"],
    "sgen": {"p_mw": base_ts["sgen"]["p_mw"] * 0.0, "q_mvar": base_ts["sgen"]["q_mvar"] * 0.0},
}

for eq in ["load", "sgen"]:
    pp_sim.apply_power_profile(net=net_load, equipment=eq, power_profiles=base_ts_zero[eq])

pp_sim.create_output_writer(net=net_load, add_results=["res_line.loading_percent"])
result_df_load = pp_sim.run_time_simulation(net=net_load)

# --- 2) PV case (200 kW) baseline ---
net_pv = copy.deepcopy(net_trey)
reset_timeseries(net_pv, drop_sgen=True)

net_pv.load["profile_mapping"] = net_pv.load["bus"]
pp.create_sgen(net_pv, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
pp.create_sgen(net_pv, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
net_pv.sgen["profile_mapping"] = net_pv.sgen["bus"]

for eq in ["load", "sgen"]:
    pp_sim.apply_power_profile(net=net_pv, equipment=eq, power_profiles=base_ts[eq])

pp_sim.create_output_writer(net=net_pv, add_results=["res_line.loading_percent"])
result_df_pv = pp_sim.run_time_simulation(net=net_pv)

# --- Target line for curtailment (specified) ---
target_line_name = "GKN3X50_50_2"  # line connected to N1

l_load_all = result_df_load["res_line.loading_percent"].copy()
l_pv_all = result_df_pv["res_line.loading_percent"].copy()

# Normalize columns to line names
line_name_map = net_pv.line["name"].to_dict()  # index -> name
l_load_all = l_load_all.rename(columns=line_name_map)
l_pv_all = l_pv_all.rename(columns=line_name_map)

# Select target column by name
if target_line_name in l_pv_all.columns:
    load_series = l_load_all[target_line_name]
    pv_series = l_pv_all[target_line_name]
else:
    # fallback: line with highest loading in PV case
    fallback_col = l_pv_all.max().idxmax()
    load_series = l_load_all[fallback_col]
    pv_series = l_pv_all[fallback_col]

# Curtail only when PV creates overload on target line
overload_mask = (pv_series > 100) & (load_series <= 100)

# --- Apply cap to N1 only on those time steps ---
reset_timeseries(net_trey, drop_sgen=True)

time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)
net_trey.load["profile_mapping"] = net_trey.load["bus"]

bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])
pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW_curt")
pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]

pv_p_base = time_series["sgen"]["p_mw"].copy()
pv_q_base = time_series["sgen"]["q_mvar"].copy()

pv_p_curt = pv_p_base.copy()
pv_q_curt = pv_q_base.copy()
if bus_n1 in pv_p_curt.columns:
    mask = overload_mask.values
    col_idx = pv_p_curt.columns.get_loc(bus_n1)
    pv_p_curt.iloc[mask, col_idx] = pv_p_curt.iloc[mask, col_idx].clip(upper=p_max_mw)
    base_vals = pv_p_base.iloc[mask, col_idx].replace(0, 1e-9)
    ratio = pv_p_curt.iloc[mask, col_idx] / base_vals
    pv_q_curt.iloc[mask, col_idx] = pv_q_base.iloc[mask, col_idx] * ratio

time_series["sgen"] = {"p_mw": pv_p_curt, "q_mvar": pv_q_curt}

for eq in ["load", "sgen"]:
    pp_sim.apply_power_profile(net=net_trey, equipment=eq, power_profiles=time_series[eq])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.loading_percent", "res_line.p_from_mw", "res_bus.vm_pu"])
result_df_curt = pp_sim.run_time_simulation(net=net_trey)

# Plots
pp_plot.plot_timeseries_result(
    data_df=result_df_curt["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="V bus – curtailment",
    filename="pv_200kw_curt_voltage",
)

pp_plot.plot_timeseries_result(
    data_df=result_df_curt["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power – curtailment",
    filename="pv_200kw_curt_line_power",
)

pp_plot.plot_timeseries_result(
    data_df=result_df_curt["res_line.loading_percent"],
    ylabel="[%]",
    plot_title="Line loading – curtailment",
    filename="pv_200kw_curt_line_loading",
)

pp_plot.plot_timestamps_powerflow_result(
    net=net_trey,
    plot_time=time(hour=12, minute=0),
    filename="pv_200kw_curt_power_flow_at_12h00",
)

# Economics
summary_curt = summarize_timeseries(result_df_curt)
energy_base_mwh = profile_energy_mwh(pv_p_base[[bus_n1]])
energy_curt_mwh = profile_energy_mwh(pv_p_curt[[bus_n1]])
energy_lost_mwh = energy_base_mwh - energy_curt_mwh
loss_chf = energy_lost_mwh * price_chf_per_mwh if price_chf_per_mwh is not None else None

overload_hours = overload_mask.sum()

{
    "p_max_mw": p_max_mw,
    "overload_hours": overload_hours,
    "summary": summary_curt,
    "energy_lost_mwh": energy_lost_mwh,
    "loss_chf": loss_chf,
}


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



/tmp/ipykernel_86310/1488186706.py:47: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipykernel_86310/1488186706.py:47: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



{'p_max_mw': 0.11777945491468367,
 'overload_hours': 43,
 'summary': {'vmin': 0.912867163699566,
  'vmax': 1.011170928589216,
  'line_loading_max': 96.31872335862978},
 'energy_lost_mwh': 1.7595617632450622,
 'loss_chf': 503.2346642880878}

In [335]:
### PV curtailment – exact energy loss from profile (summer week)
import pandas as pd
import numpy as np

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])

# Read sgen sheet
df_sgen = pd.read_excel(profile_file_path, sheet_name="sgen", header=[0,1])
time_col = df_sgen.columns[0]
time_vals = pd.to_datetime(df_sgen[time_col].astype(str))
dt = time_vals.diff().dt.total_seconds().fillna(time_vals.diff().dt.total_seconds().median()).fillna(3600)
dt_hours = (dt.to_numpy() / 3600.0)

# Extract PV P profile at N1 (MW)
p_col = (bus_n1, "P [MW]")
if p_col not in df_sgen.columns:
    for c in df_sgen.columns:
        if str(c[0]) == str(bus_n1) and c[1] == "P [MW]":
            p_col = c
            break

pv_p_mw = df_sgen[p_col].fillna(0.0).to_numpy()

# Cap (MW)
pmax_mw = 0.118  # 118 kW (400 V, 170 A)

def energy_lost_mwh(p_mw, cap_mw, dt_h):
    lost = np.maximum(0.0, p_mw - cap_mw)
    return float(np.sum(lost * dt_h))

E_lost = energy_lost_mwh(pv_p_mw, pmax_mw, dt_hours)

price_chf_per_kwh = 0.0952
price_chf_per_mwh = price_chf_per_kwh * 1000

N_summer_days = 123

{
    "E_lost_MWh_per_day": E_lost,
    "Cost_CHF_per_day": E_lost * price_chf_per_mwh,
    "E_lost_MWh_summer_period": E_lost * N_summer_days,
    "Cost_CHF_summer_period": E_lost * price_chf_per_mwh * N_summer_days,
}


/tmp/ipykernel_86310/3381591885.py:11: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



{'E_lost_MWh_per_day': 1.77752397708458,
 'Cost_CHF_per_day': 169.22028261845202,
 'E_lost_MWh_summer_period': 218.63544918140335,
 'Cost_CHF_summer_period': 20814.0947620696}

In [336]:
### PV 200 kW – cos(phi) sensitivity (summer week)
# cosφ = 0.9 and 0.8 on N1 (Q absorbed to mitigate overvoltage)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
cosphi_values = [0.9, 0.8]
mode = "absorb"  # 'absorb' -> Q negative, 'inject' -> Q positive

def _coerce_cols_to_int(df):
    try:
        df.columns = df.columns.astype(int)
    except Exception:
        try:
            df.columns = [int(c) if str(c).isdigit() else c for c in df.columns]
        except Exception:
            pass
    return df

for cosphi in cosphi_values:
    reset_timeseries(net_trey, drop_sgen=True)
    time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

    net_trey.load["profile_mapping"] = net_trey.load["bus"]
    bus_n1 = int(net_trey.bus.index[net_trey.bus["name"] == "N1"][0])
    bus_60437 = int(net_trey.bus.index[net_trey.bus["name"] == "60437"][0])

    pp.create_sgen(net_trey, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name=f"PV_N1_cosphi_{cosphi}")
    pp.create_sgen(net_trey, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
    net_trey.sgen["profile_mapping"] = net_trey.sgen["bus"]

    pv_p = _coerce_cols_to_int(time_series["sgen"]["p_mw"].copy())
    pv_q = _coerce_cols_to_int(time_series["sgen"]["q_mvar"].copy())

    phi = np.arccos(cosphi)
    q_sign = -1 if mode == "absorb" else 1
    if bus_n1 in pv_p.columns:
        pv_q.loc[:, bus_n1] = pv_p.loc[:, bus_n1] * np.tan(phi) * q_sign
    else:
        print(f"[WARN] N1 column {bus_n1} not found in sgen profile columns: {list(pv_p.columns)[:6]}...")

    time_series["sgen"] = {"p_mw": pv_p, "q_mvar": pv_q}

    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net_trey, equipment=eq, power_profiles=time_series[eq])

    pp_sim.create_output_writer(net=net_trey, add_results=["res_bus.vm_pu", "res_line.loading_percent"])
    result_df_cosphi = pp_sim.run_time_simulation(net=net_trey)

    pp_plot.plot_timeseries_result(
        data_df=result_df_cosphi["res_bus.vm_pu"],
        ylabel="V [pu]",
        plot_title=f"Bus voltage (cosφ={cosphi}, {mode})",
        filename=f"pv_200kw_cosphi_{cosphi}_{mode}_voltage",
    )

    pp_plot.plot_timeseries_result(
        data_df=result_df_cosphi["res_line.loading_percent"],
        ylabel="[%]",
        plot_title=f"Line loading (cosφ={cosphi}, {mode} – summer week)",
        filename=f"pv_200kw_cosphi_{cosphi}_{mode}_line_loading",
    )


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



In [337]:
### cos(phi) – Joule losses and economic impact (summer week)
import copy
import numpy as np
import pandas as pd

price_chf_per_kwh = 0.1455
price_chf_per_mwh = price_chf_per_kwh * 1000

def _dt_hours(index):
    try:
        idx = pd.to_datetime(index)
        dt = pd.Series(idx).diff().dt.total_seconds().to_numpy()
        if len(dt) == 0:
            return np.array([1.0])
        med = pd.Series(dt[1:]).replace(0, np.nan).median() if len(dt) > 1 else dt[0]
        if not med or med <= 0:
            med = 3600
        dt[0] = med
        dt = np.where(dt == 0, med, dt)
        return dt / 3600.0
    except Exception:
        return np.ones(len(index))

def _find_profile_col(df, bus):
    if bus in df.columns:
        return bus
    for c in df.columns:
        if str(c) == str(bus):
            return c
    return None

try:
    _coerce_cols_to_int
except NameError:
    def _coerce_cols_to_int(df):
        try:
            df.columns = df.columns.astype(int)
        except Exception:
            try:
                df.columns = [int(c) if str(c).isdigit() else c for c in df.columns]
            except Exception:
                pass
        return df

def run_cosphi_losses(cosphi):
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)
    ts = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name=f"PV_N1_cosphi_{cosphi}")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
    net.sgen["profile_mapping"] = net.sgen["bus"]

    p_sgen = _coerce_cols_to_int(ts["sgen"]["p_mw"].copy())
    q_sgen = _coerce_cols_to_int(ts["sgen"]["q_mvar"].copy())

    col_n1 = _find_profile_col(p_sgen, bus_n1)
    if col_n1 is not None and cosphi < 1.0:
        q_sgen[col_n1] = -p_sgen[col_n1] * np.tan(np.arccos(cosphi))

    ts["sgen"] = {"p_mw": p_sgen, "q_mvar": q_sgen}
    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=ts[eq])

    pp_sim.create_output_writer(net=net, add_results=["res_line.pl_mw"])
    result = pp_sim.run_time_simulation(net=net)

    pl_mw = result["res_line.pl_mw"]
    dt_hours = _dt_hours(pl_mw.index)
    e_loss_mwh = (pl_mw.sum(axis=1).to_numpy() * dt_hours).sum()
    return e_loss_mwh

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
E_loss_1 = run_cosphi_losses(1.0)
E_loss_09 = run_cosphi_losses(0.9)
E_loss_08 = run_cosphi_losses(0.8)

delta_09 = E_loss_09 - E_loss_1
delta_08 = E_loss_08 - E_loss_1

cost_09_chf = delta_09 * price_chf_per_mwh
cost_08_chf = delta_08 * price_chf_per_mwh

{
    "E_loss_cosphi_1_MWh": E_loss_1,
    "E_loss_cosphi_0.9_MWh": E_loss_09,
    "E_loss_cosphi_0.8_MWh": E_loss_08,
    "Delta_loss_0.9_MWh": delta_09,
    "Delta_loss_0.8_MWh": delta_08,
    "Cost_0.9_CHF_per_day": cost_09_chf,
    "Cost_0.8_CHF_per_day": cost_08_chf,
}


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'E_loss_cosphi_1_MWh': 2.24539831973759,
 'E_loss_cosphi_0.9_MWh': 2.8602353030498073,
 'E_loss_cosphi_0.8_MWh': 3.687550966430182,
 'Delta_loss_0.9_MWh': 0.6148369833122174,
 'Delta_loss_0.8_MWh': 1.442152646692592,
 'Cost_0.9_CHF_per_day': 89.45878107192763,
 'Cost_0.8_CHF_per_day': 209.83321009377215}

In [338]:
### PV 200 kW – line replacement cost (based on limiting line)
# Estimate cost for reinforcing the limiting line
cost_per_km_chf = 108940  # CHF/km (tripolaire 240 mm2: 10894/100m)

# Determine limiting line if not already defined
if "limiting_line" not in globals():
    if "result_df_200kw" in globals():
        l_tmp = result_df_200kw["res_line.loading_percent"]
        limiting_line = l_tmp.max().idxmax()
    else:
        raise NameError("limiting_line not defined: run PV 200 kW simulation first")

# Try to map limiting line to net_trey.line
line_name = limiting_line
length_km = None
if line_name in net_trey.line.index:
    length_km = net_trey.line.at[line_name, "length_km"]
else:
    match = net_trey.line[net_trey.line["name"] == line_name]
    if len(match):
        length_km = match.iloc[0]["length_km"]

cost_chf = length_km * cost_per_km_chf if length_km is not None else None
{"limiting_line": line_name, "length_km": length_km, "cost_chf": cost_chf}



{'limiting_line': 'GKN3X50_50_2', 'length_km': 0.255, 'cost_chf': 27779.7}

**Analyse – remplacement de ligne (cout)**
Le cout du cable 240 mm2 tripolaire est de 10'894 CHF / 100 m, soit 108'940 CHF / km. Il faut ajouter les travaux de genie civil (fouille, pose, jonctions), souvent preponderants. Ce point justifie l’analyse economique detaillee.


In [339]:
### PV 200 kW – line replacement (simulation, improved)

# Helper: find line index by name or index

def _get_line_idx(net, line_name):
    if line_name in net.line.index:
        return line_name
    match = net.line[net.line["name"] == line_name]
    if len(match):
        return match.index[0]
    raise KeyError(f"Line {line_name} not found in net")

# Helper: upgrade one line to GKN3X240_240

def _upgrade_to_240(net, line_idx):
    upgrade_params = {
        "r_ohm_per_km": 0.0754,
        "x_ohm_per_km": 0.07,
        "c_nf_per_km": 346,
        "max_i_ka": 0.512,
    }
    for k, v in upgrade_params.items():
        net.line.at[line_idx, k] = v
    net.line.at[line_idx, "name"] = f"{net.line.at[line_idx, 'name']}_240"

# Helper: run 200 kW scenario on a given net

def _run_200kw(net):
    reset_timeseries(net, drop_sgen=True)
    profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
    time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
    net.sgen["profile_mapping"] = net.sgen["bus"]

    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=time_series[eq])

    pp_sim.create_output_writer(net=net, add_results=["res_line.loading_percent", "res_line.p_from_mw", "res_bus.vm_pu"])
    result_df = pp_sim.run_time_simulation(net=net)
    l = result_df["res_line.loading_percent"]
    overloaded = list(l.columns[(l > 100).any()].values)
    return result_df, overloaded

# Baseline on a copy
net_reinf = copy.deepcopy(net_trey)
result_df_base, overloaded_base = _run_200kw(net_reinf)

# 1) Upgrade all overloaded lines to 240
for ln in overloaded_base:
    line_idx = _get_line_idx(net_reinf, ln)
    _upgrade_to_240(net_reinf, line_idx)

result_df_reinf, overloaded_after = _run_200kw(net_reinf)

# 2) If still overloaded, add parallels (ceil)
# target max loading (safety margin)
target_loading = 90

if overloaded_after:
    l_after = result_df_reinf["res_line.loading_percent"]
    for ln in overloaded_after:
        line_idx = _get_line_idx(net_reinf, ln)
        max_load = l_after[ln].max()
        net_reinf.line.at[line_idx, "parallel"] = int(np.ceil(max_load / target_loading))
    result_df_reinf, overloaded_after = _run_200kw(net_reinf)

# Plots (after reinforcement)
pp_plot.plot_timeseries_result(
    data_df=result_df_reinf["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage (reinforced lines)",
    filename="pv_200kw_reinf_voltage",
)

pp_plot.plot_timeseries_result(
    data_df=result_df_reinf["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Line power (reinforced lines)",
    filename="pv_200kw_reinf_line_power",
)

pp_plot.plot_timeseries_result(
    data_df=result_df_reinf["res_line.loading_percent"],
    ylabel="[%]",
    plot_title="Line loading (reinforced lines)",
    filename="pv_200kw_reinf_line_loading",
)

pp_plot.plot_timestamps_powerflow_result(
    net=net_reinf,
    plot_time=time(hour=12, minute=0),
    filename="pv_200kw_reinf_power_flow_at_12h00",
)

{
    "overloaded_lines_before": overloaded_base,
    "overloaded_lines_after": overloaded_after,
    "line_loading_max_after": result_df_reinf["res_line.loading_percent"].max().max(),
    "target_loading": target_loading,
    "vmin_after": result_df_reinf["res_bus.vm_pu"].min().min(),
    "vmax_after": result_df_reinf["res_bus.vm_pu"].max().max(),
}


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'overloaded_lines_before': ['GKN3x150_150_3', 'GKN3X50_50_2'],
 'overloaded_lines_after': [],
 'line_loading_max_after': 95.19909741461714,
 'target_loading': 90,
 'vmin_after': 0.912866199990704,
 'vmax_after': 1.0}

In [340]:
### PV 200 kW – parallel line on GKN3X50_50_2 (simulation)
import copy

# fallback helpers if not already defined
try:
    _get_line_idx
except NameError:
    def _get_line_idx(net, line_name):
        if line_name in net.line.index:
            return line_name
        match = net.line[net.line["name"] == line_name]
        if len(match):
            return match.index[0]
        raise KeyError(f"Line {line_name} not found in net")

try:
    _run_200kw
except NameError:
    def _run_200kw(net):
        reset_timeseries(net, drop_sgen=True)
        profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
        time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

        net.load["profile_mapping"] = net.load["bus"]
        bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
        bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
        pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
        pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
        net.sgen["profile_mapping"] = net.sgen["bus"]

        for eq in ["load", "sgen"]:
            pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=time_series[eq])

        pp_sim.create_output_writer(net=net, add_results=["res_line.loading_percent", "res_line.p_from_mw", "res_bus.vm_pu"])
        result_df = pp_sim.run_time_simulation(net=net)
        l = result_df["res_line.loading_percent"]
        overloaded = list(l.columns[(l > 100).any()].values)
        return result_df, overloaded

# Build parallel-line case
net_parallel = copy.deepcopy(net_trey)
line_name = "GKN3X50_50_2"
line_idx = _get_line_idx(net_parallel, line_name)
net_parallel.line.at[line_idx, "parallel"] = 2

result_df_parallel, overloaded_parallel = _run_200kw(net_parallel)

# Plots (parallel line)
pp_plot.plot_timeseries_result(
    data_df=result_df_parallel["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage (parallel line)",
    filename="pv_200kw_parallel_voltage",
)

pp_plot.plot_timeseries_result(
    data_df=result_df_parallel["res_line.loading_percent"],
    ylabel="[%]",
    plot_title="Line loading (parallel line)",
    filename="pv_200kw_parallel_line_loading",
)

pp_plot.plot_timestamps_powerflow_result(
    net=net_parallel,
    plot_time=time(hour=12, minute=0),
    filename="pv_200kw_parallel_power_flow_at_12h00",
)

{
    "overloaded_lines_parallel": overloaded_parallel,
    "line_loading_max_parallel": result_df_parallel["res_line.loading_percent"].max().max(),
    "vmin_parallel": result_df_parallel["res_bus.vm_pu"].min().min(),
    "vmax_parallel": result_df_parallel["res_bus.vm_pu"].max().max(),
}


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'overloaded_lines_parallel': ['GKN3x150_150_3', 'GKN3X50_50_2'],
 'line_loading_max_parallel': 140.5822360117242,
 'vmin_parallel': 0.912867524169902,
 'vmax_parallel': 1.0623675332668445}

## 9. Tap


In [341]:
### Transformer tap ±5% and -6.885% (time-series check)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

def run_tap_case(tap_delta_percent):
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)

    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
    net.sgen["profile_mapping"] = net.sgen["bus"]

    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=time_series[eq])

    trafo_idx = net.trafo.index[0]
    tap_step = net.trafo.at[trafo_idx, "tap_step_percent"]
    tap_orig = net.trafo.at[trafo_idx, "tap_pos"]
    if pd.isna(tap_step) or tap_step == 0:
        tap_step = 2.295
    delta_pos = int(round(tap_delta_percent / tap_step))
    net.trafo.at[trafo_idx, "tap_pos"] = tap_orig + delta_pos

    pp_sim.create_output_writer(net=net, add_results=["res_bus.vm_pu", "res_line.loading_percent"])
    return pp_sim.run_time_simulation(net=net)

res_tap_m6885 = run_tap_case(-6.885)
res_tap_m5 = run_tap_case(-5.0)
res_tap_0 = run_tap_case(0.0)
res_tap_p5 = run_tap_case(5.0)

# Vmax comparison only
vmax_df = pd.DataFrame({
    "tap -6.885%": res_tap_m6885["res_bus.vm_pu"].max(axis=1),
    "tap -5%": res_tap_m5["res_bus.vm_pu"].max(axis=1),
    "tap 0%": res_tap_0["res_bus.vm_pu"].max(axis=1),
    "tap +5%": res_tap_p5["res_bus.vm_pu"].max(axis=1),
})
pp_plot.plot_timeseries_result(
    data_df=vmax_df,
    ylabel="Vmax [pu]",
    plot_title="Maximum bus voltage vs tap position",
    filename="tap_vmax",
)

# Detailed V bus only for -6.885% and -5%
pp_plot.plot_timeseries_result(
    data_df=res_tap_m6885["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage – tap -6.885%",
    filename="tap_m6885_bus_voltage",
)
pp_plot.plot_timeseries_result(
    data_df=res_tap_m5["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Bus voltage – tap -5%",
    filename="tap_m5_bus_voltage",
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/l

In [342]:
### Tap – Joule losses and economic impact (summer week)
import copy
import numpy as np
import pandas as pd

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
price_chf_per_kwh = 0.1455
price_chf_per_mwh = price_chf_per_kwh * 1000
N_summer_days = 123

def _dt_hours(index):
    try:
        idx = pd.to_datetime(index)
        dt = pd.Series(idx).diff().dt.total_seconds().to_numpy()
        if len(dt) == 0:
            return np.array([1.0])
        med = pd.Series(dt[1:]).replace(0, np.nan).median() if len(dt) > 1 else dt[0]
        if not med or med <= 0:
            med = 3600
        dt[0] = med
        dt = np.where(dt == 0, med, dt)
        return dt / 3600.0
    except Exception:
        return np.ones(len(index))

def run_tap_case_loss(tap_delta_percent):
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)
    time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1_200kW")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437_base")
    net.sgen["profile_mapping"] = net.sgen["bus"]

    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=time_series[eq])

    trafo_idx = net.trafo.index[0]
    tap_step = net.trafo.at[trafo_idx, "tap_step_percent"]
    tap_orig = net.trafo.at[trafo_idx, "tap_pos"]
    if pd.isna(tap_step) or tap_step == 0:
        tap_step = 2.295
    delta_pos = int(round(tap_delta_percent / tap_step))
    net.trafo.at[trafo_idx, "tap_pos"] = tap_orig + delta_pos

    pp_sim.create_output_writer(net=net, add_results=["res_line.pl_mw"])
    result_df = pp_sim.run_time_simulation(net=net)
    pl_mw = result_df["res_line.pl_mw"]
    dt_hours = _dt_hours(pl_mw.index)
    e_loss_mwh = (pl_mw.sum(axis=1).to_numpy() * dt_hours).sum()
    return e_loss_mwh

E_tap_0 = run_tap_case_loss(0.0)
E_tap_m5 = run_tap_case_loss(-5.0)
E_tap_m6885 = run_tap_case_loss(-6.885)
E_tap_p5 = run_tap_case_loss(5.0)

def _delta(e):
    return e - E_tap_0

out = {
    "E_loss_tap_0_MWh_per_day": E_tap_0,
    "E_loss_tap_-5_MWh_per_day": E_tap_m5,
    "E_loss_tap_-6.885_MWh_per_day": E_tap_m6885,
    "E_loss_tap_+5_MWh_per_day": E_tap_p5,
    "Delta_-5_MWh_per_day": _delta(E_tap_m5),
    "Delta_-6.885_MWh_per_day": _delta(E_tap_m6885),
    "Delta_+5_MWh_per_day": _delta(E_tap_p5),
    "Cost_-5_CHF_per_day": _delta(E_tap_m5) * price_chf_per_mwh,
    "Cost_-6.885_CHF_per_day": _delta(E_tap_m6885) * price_chf_per_mwh,
    "Cost_+5_CHF_per_day": _delta(E_tap_p5) * price_chf_per_mwh,
    "Cost_-5_CHF_summer": _delta(E_tap_m5) * price_chf_per_mwh * N_summer_days,
    "Cost_-6.885_CHF_summer": _delta(E_tap_m6885) * price_chf_per_mwh * N_summer_days,
    "Cost_+5_CHF_summer": _delta(E_tap_p5) * price_chf_per_mwh * N_summer_days,
}
out


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/l

{'E_loss_tap_0_MWh_per_day': 2.24539831973759,
 'E_loss_tap_-5_MWh_per_day': 2.28449183964924,
 'E_loss_tap_-6.885_MWh_per_day': 2.3061801726285767,
 'E_loss_tap_+5_MWh_per_day': 2.2115054490183956,
 'Delta_-5_MWh_per_day': 0.03909351991165,
 'Delta_-6.885_MWh_per_day': 0.0607818528909867,
 'Delta_+5_MWh_per_day': -0.033892870719194335,
 'Cost_-5_CHF_per_day': 5.688107147145075,
 'Cost_-6.885_CHF_per_day': 8.843759595638565,
 'Cost_+5_CHF_per_day': -4.931412689642776,
 'Cost_-5_CHF_summer': 699.6371790988443,
 'Cost_-6.885_CHF_summer': 1087.7824302635433,
 'Cost_+5_CHF_summer': -606.5637608260614}

## Battery

In [343]:
### Storage sizing (from chosen battery)
# Battery tech (Table 1)
storage_cost_per_kwh = 1000  # CHF/kWh
storage_eff = 0.90
storage_cycles = 10000

# Chosen battery size (from simulations)
storage_power_mw = 0.3  # 300 kW
storage_energy_mwh = 0.6  # 600 kWh

storage_energy_kwh = storage_energy_mwh * 1000
storage_cost_chf = storage_energy_kwh * storage_cost_per_kwh

lifetime_throughput_kwh = storage_energy_kwh * storage_cycles * storage_eff
lcos_chf_per_kwh = storage_cost_chf / lifetime_throughput_kwh

{
    "storage_power_kw": storage_power_mw * 1000,
    "storage_energy_kwh": storage_energy_kwh,
    "storage_cost_chf": storage_cost_chf,
    "lcos_chf_per_kwh": lcos_chf_per_kwh,
}


{'storage_power_kw': 300.0,
 'storage_energy_kwh': 600.0,
 'storage_cost_chf': 600000.0,
 'lcos_chf_per_kwh': 0.1111111111111111}

In [344]:
### Storage – simulation (charge to limit overvoltage)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
base_ts = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Storage parameters (chosen size)
storage_eff = 0.90
storage_energy_mwh = 0.6  # 600 kWh
storage_power_mw = 0.3  # 300 kW
vmax_limit = 1.05  # pu

# Time step (hours)
idx = base_ts["sgen"]["p_mw"].index
if len(idx) >= 2:
    try:
        dt_h = (idx[1] - idx[0]).total_seconds() / 3600
    except Exception:
        dt_h = 1.0
else:
    dt_h = 1.0

def _pick_n1_col(df, bus_n1):
    if bus_n1 in df.columns:
        return bus_n1
    for c in df.columns:
        if str(c) == str(bus_n1):
            return c
    if len(df.columns) >= 1:
        print("[WARN] N1 column not found; using first sgen column as fallback.")
        return df.columns[0]
    return None

def build_net_with_pv():
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)
    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437")
    net.sgen["profile_mapping"] = net.sgen["bus"]
    return net, bus_n1

def run_ts(net, ts, add_results):
    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=ts[eq])
    pp_sim.create_output_writer(net=net, add_results=add_results)
    return pp_sim.run_time_simulation(net=net)

# Baseline PV run to detect overvoltage
net_pv, bus_n1 = build_net_with_pv()
result_df_pv = run_ts(net_pv, base_ts, ["res_bus.vm_pu"])
vmax_series = result_df_pv["res_bus.vm_pu"].max(axis=1)
overvoltage_mask = vmax_series > vmax_limit
overvoltage_mask = overvoltage_mask.reindex(base_ts["sgen"]["p_mw"].index, fill_value=False)

# Build storage-charged PV profile
pv_p = base_ts["sgen"]["p_mw"].copy()
pv_q = base_ts["sgen"]["q_mvar"].copy()
pv_p_storage = pv_p.copy()
pv_q_storage = pv_q.copy()

col_n1 = _pick_n1_col(pv_p_storage, bus_n1)
if col_n1 is None:
    raise RuntimeError("N1 column not found in sgen profile.")

soc_mwh = 0.0
p_charge_profile = []
soc_profile = []

for t in range(len(pv_p_storage)):
    p_charge = 0.0
    if overvoltage_mask.iloc[t]:
        p_avail = pv_p_storage.iloc[t, pv_p_storage.columns.get_loc(col_n1)]
        p_charge = min(storage_power_mw, p_avail, max(0.0, (storage_energy_mwh - soc_mwh) / dt_h))
        pv_p_storage.iloc[t, pv_p_storage.columns.get_loc(col_n1)] -= p_charge
        soc_mwh += p_charge * dt_h * storage_eff
    p_charge_profile.append(p_charge)
    soc_profile.append(soc_mwh)

base_vals = pv_p[col_n1].replace(0, 1e-9)
ratio = pv_p_storage[col_n1] / base_vals
pv_q_storage[col_n1] = pv_q[col_n1] * ratio

ts_storage = {
    "load": base_ts["load"],
    "sgen": {"p_mw": pv_p_storage, "q_mvar": pv_q_storage},
}

net_storage, _ = build_net_with_pv()
result_df_storage = run_ts(net_storage, ts_storage, ["res_bus.vm_pu", "res_line.loading_percent"])

# Plots: network impact + storage behavior
pp_plot.plot_timeseries_result(
    data_df=result_df_storage["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="V bus – storage (Vmax)",
    filename="storage_bus_voltage",
)
pp_plot.plot_timeseries_result(
    data_df=result_df_storage["res_line.loading_percent"],
    ylabel="[%]",
    plot_title="Line loading – storage (Vmax)",
    filename="storage_line_loading",
)

storage_df = pd.DataFrame({
    "P_charge_MW": p_charge_profile,
    "SOC_MWh": soc_profile,
}, index=base_ts["sgen"]["p_mw"].index)

pp_plot.plot_timeseries_result(
    data_df=storage_df[["P_charge_MW"]],
    ylabel="P [MW]",
    plot_title="Storage charging power (N1)",
    filename="storage_charging_power",
)
pp_plot.plot_timeseries_result(
    data_df=storage_df[["SOC_MWh"]],
    ylabel="Energy [MWh]",
    plot_title="Storage SOC (N1)",
    filename="storage_soc",
)

{
    "vmax_limit": vmax_limit,
    "storage_energy_mwh": storage_energy_mwh,
    "storage_power_mw": storage_power_mw,
    "final_soc_mwh": soc_mwh,
}


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



{'vmax_limit': 1.05,
 'storage_energy_mwh': 0.6,
 'storage_power_mw': 0.3,
 'final_soc_mwh': 0.6}

In [345]:
### Storage + transformer tap (overvoltage control)

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
base_ts = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Storage parameters
storage_eff = 0.90
storage_energy_mwh = 0.6  # 600 kWh
storage_power_mw = 0.3  # 300 kW
vmax_limit = 1.05
tap_values = [-5.0, -6.885]  # [%]

# Time step (hours)
idx = base_ts["sgen"]["p_mw"].index
if len(idx) >= 2:
    try:
        dt_h = (idx[1] - idx[0]).total_seconds() / 3600
    except Exception:
        dt_h = 1.0
else:
    dt_h = 1.0

def build_net_with_pv():
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)
    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437")
    net.sgen["profile_mapping"] = net.sgen["bus"]
    return net, bus_n1

def apply_tap(net, tap_delta_percent):
    trafo_idx = net.trafo.index[0]
    tap_step = net.trafo.at[trafo_idx, "tap_step_percent"]
    tap_orig = net.trafo.at[trafo_idx, "tap_pos"]
    if pd.isna(tap_step) or tap_step == 0:
        tap_step = 2.295
    delta_pos = int(round(tap_delta_percent / tap_step))
    net.trafo.at[trafo_idx, "tap_pos"] = tap_orig + delta_pos

def run_ts(net, ts, add_results):
    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=ts[eq])
    pp_sim.create_output_writer(net=net, add_results=add_results)
    return pp_sim.run_time_simulation(net=net)

results = {}
for tap in tap_values:
    # Baseline PV run with tap -> detect overvoltage
    net_pv, bus_n1 = build_net_with_pv()
    apply_tap(net_pv, tap)
    res_pv = run_ts(net_pv, base_ts, ["res_bus.vm_pu"])
    vmax_series = res_pv["res_bus.vm_pu"].max(axis=1)
    overvoltage_mask = (vmax_series > vmax_limit).reindex(base_ts["sgen"]["p_mw"].index, fill_value=False)

    # Build storage-charged PV profile
    pv_p = base_ts["sgen"]["p_mw"].copy()
    pv_q = base_ts["sgen"]["q_mvar"].copy()
    pv_p_storage = pv_p.copy()
    pv_q_storage = pv_q.copy()

    soc_mwh = 0.0
    if bus_n1 in pv_p_storage.columns:
        for t in range(len(pv_p_storage)):
            if overvoltage_mask.iloc[t]:
                p_avail = pv_p_storage.iloc[t, pv_p_storage.columns.get_loc(bus_n1)]
                p_charge = min(storage_power_mw, p_avail, max(0.0, (storage_energy_mwh - soc_mwh) / dt_h))
                pv_p_storage.iloc[t, pv_p_storage.columns.get_loc(bus_n1)] -= p_charge
                soc_mwh += p_charge * dt_h * storage_eff
        base_vals = pv_p.iloc[:, pv_p.columns.get_loc(bus_n1)].replace(0, 1e-9)
        ratio = pv_p_storage.iloc[:, pv_p_storage.columns.get_loc(bus_n1)] / base_vals
        pv_q_storage.iloc[:, pv_q.columns.get_loc(bus_n1)] = pv_q.iloc[:, pv_q.columns.get_loc(bus_n1)] * ratio

    ts_storage = {
        "load": base_ts["load"],
        "sgen": {"p_mw": pv_p_storage, "q_mvar": pv_q_storage},
    }

    net_storage, _ = build_net_with_pv()
    apply_tap(net_storage, tap)
    res_storage = run_ts(net_storage, ts_storage, ["res_bus.vm_pu", "res_line.loading_percent"])
    results[tap] = res_storage

# Plots: Vmax comparison + detailed V bus for each tap
vmax_df = pd.DataFrame({
    f"tap {tap}%": res["res_bus.vm_pu"].max(axis=1) for tap, res in results.items()
})
pp_plot.plot_timeseries_result(
    data_df=vmax_df,
    ylabel="Vmax [pu]",
    plot_title="Vmax – storage + tap",
    filename="storage_tap_vmax",
)

for tap, res in results.items():
    pp_plot.plot_timeseries_result(
        data_df=res["res_bus.vm_pu"],
        ylabel="V [pu]",
        plot_title=f"Bus voltage – storage + tap {tap}%",
        filename=f"storage_tap_bus_voltage_{str(tap).replace('.', 'p')}",
    )


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/l

In [346]:
### Combo: tap -5% + curtailment (Vmax) + storage

profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week_with_prod_200kW.xlsx"
base_ts = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Parameters
tap_delta = -5.0
I_ka = 0.17
U_ll_kv = 0.4
cosphi = 1.0
p_max_mw = (np.sqrt(3) * U_ll_kv * I_ka * cosphi)

storage_eff = 0.90
storage_power_mw = 0.3
storage_energy_mwh = 0.6
vmax_limit = 1.05

# Time step (hours)
idx = base_ts["sgen"]["p_mw"].index
if len(idx) >= 2:
    try:
        dt_h = (idx[1] - idx[0]).total_seconds() / 3600
    except Exception:
        dt_h = 1.0
else:
    dt_h = 1.0

def _coerce_cols_to_int(df):
    try:
        df.columns = df.columns.astype(int)
    except Exception:
        try:
            df.columns = [int(c) if str(c).isdigit() else c for c in df.columns]
        except Exception:
            pass
    return df

def _find_profile_col(df, bus_n1, net):
    candidates = [bus_n1, str(bus_n1)]
    if "bus" in net.sgen.columns:
        sgen_idx = net.sgen.index[net.sgen["bus"] == bus_n1]
        if len(sgen_idx):
            candidates += [sgen_idx[0], str(sgen_idx[0])]
    if "name" in net.sgen.columns:
        idx_name = net.sgen.index[net.sgen["name"].astype(str).str.contains("N1", case=False, na=False)]
        if len(idx_name):
            candidates += [idx_name[0], str(idx_name[0])]
    for c in candidates:
        if c in df.columns:
            return c
    cols_int = {}
    for col in df.columns:
        try:
            cols_int[int(col)] = col
        except Exception:
            pass
    return cols_int.get(bus_n1, None)

def build_net_with_pv():
    net = copy.deepcopy(net_trey)
    reset_timeseries(net, drop_sgen=True)
    net.load["profile_mapping"] = net.load["bus"]
    bus_n1 = int(net.bus.index[net.bus["name"] == "N1"][0])
    bus_60437 = int(net.bus.index[net.bus["name"] == "60437"][0])
    pp.create_sgen(net, bus=bus_n1, p_mw=0.0, q_mvar=0.0, name="PV_N1")
    pp.create_sgen(net, bus=bus_60437, p_mw=0.0, q_mvar=0.0, name="PV_60437")
    net.sgen["profile_mapping"] = net.sgen["bus"]
    return net, bus_n1

def apply_tap(net, tap_delta_percent):
    trafo_idx = net.trafo.index[0]
    tap_step = net.trafo.at[trafo_idx, "tap_step_percent"]
    tap_orig = net.trafo.at[trafo_idx, "tap_pos"]
    if pd.isna(tap_step) or tap_step == 0:
        tap_step = 2.295
    delta_pos = int(round(tap_delta_percent / tap_step))
    net.trafo.at[trafo_idx, "tap_pos"] = tap_orig + delta_pos

def run_ts(net, ts, add_results):
    for eq in ["load", "sgen"]:
        pp_sim.apply_power_profile(net=net, equipment=eq, power_profiles=ts[eq])
    pp_sim.create_output_writer(net=net, add_results=add_results)
    return pp_sim.run_time_simulation(net=net)

# 1) PV baseline with tap to get overvoltage mask
net_pv, bus_n1 = build_net_with_pv()
apply_tap(net_pv, tap_delta)
res_pv = run_ts(net_pv, base_ts, ["res_bus.vm_pu"])
vmax_series = res_pv["res_bus.vm_pu"].max(axis=1)
overvoltage_mask = (vmax_series > vmax_limit).reindex(base_ts["sgen"]["p_mw"].index, fill_value=False)

# 2) Curtailment on Vmax condition
pv_p = _coerce_cols_to_int(base_ts["sgen"]["p_mw"].copy())
pv_q = _coerce_cols_to_int(base_ts["sgen"]["q_mvar"].copy())
col_n1 = _find_profile_col(pv_p, bus_n1, net_pv)
if col_n1 is None:
    raise RuntimeError(f"N1 column not found in sgen profiles. Columns: {list(pv_p.columns)[:6]}...")

pv_p_curt = pv_p.copy()
pv_q_curt = pv_q.copy()
pv_p_curt.loc[overvoltage_mask, col_n1] = pv_p_curt.loc[overvoltage_mask, col_n1].clip(upper=p_max_mw)
base_vals = pv_p.loc[overvoltage_mask, col_n1].replace(0, 1e-9)
ratio = pv_p_curt.loc[overvoltage_mask, col_n1] / base_vals
pv_q_curt.loc[overvoltage_mask, col_n1] = pv_q.loc[overvoltage_mask, col_n1] * ratio

# 3) Storage charge after curtailment
pv_p_store = pv_p_curt.copy()
pv_q_store = pv_q_curt.copy()
soc_mwh = 0.0
p_charge_profile = []
soc_profile = []

for t in range(len(pv_p_store)):
    row = pv_p_store.index[t]
    p_charge = 0.0
    if overvoltage_mask.iloc[t]:
        p_avail = pv_p_store.loc[row, col_n1]
        p_charge = min(storage_power_mw, p_avail, max(0.0, (storage_energy_mwh - soc_mwh) / dt_h))
        pv_p_store.loc[row, col_n1] -= p_charge
        soc_mwh += p_charge * dt_h * storage_eff
    p_charge_profile.append(p_charge)
    soc_profile.append(soc_mwh)

base_vals = pv_p.loc[:, col_n1].replace(0, 1e-9)
ratio = pv_p_store.loc[:, col_n1] / base_vals
pv_q_store.loc[:, col_n1] = pv_q.loc[:, col_n1] * ratio

ts_combo = {
    "load": base_ts["load"],
    "sgen": {"p_mw": pv_p_store, "q_mvar": pv_q_store},
}

net_combo, _ = build_net_with_pv()
apply_tap(net_combo, tap_delta)
res_combo = run_ts(net_combo, ts_combo, ["res_bus.vm_pu", "res_line.loading_percent"])

# Clean bus voltage plot
bus_df = res_combo["res_bus.vm_pu"]
clean_df = pd.DataFrame({
    "Vmax": bus_df.max(axis=1),
    "Vmin": bus_df.min(axis=1),
})
bus_names = ["N1", "60437", "STMT003438HV"]
for name in bus_names:
    idx = net_combo.bus.index[net_combo.bus["name"] == name]
    if len(idx):
        b = int(idx[0])
        if b in bus_df.columns:
            clean_df[name] = bus_df[b]

pp_plot.plot_timeseries_result(
    data_df=clean_df,
    ylabel="V [pu]",
    plot_title="V bus – tap -5% + curtail + storage",
    filename="combo_tap_curt_storage_voltage_clean",
)

pp_plot.plot_timeseries_result(
    data_df=res_combo["res_line.loading_percent"],
    ylabel="[%]",
    plot_title="Line loading – tap -5% + curtail + storage",
    filename="combo_tap_curt_storage_loading",
)

storage_df = pd.DataFrame({
    "P_charge_MW": p_charge_profile,
    "SOC_MWh": soc_profile,
}, index=base_ts["sgen"]["p_mw"].index)

pp_plot.plot_timeseries_result(
    data_df=storage_df[["P_charge_MW"]],
    ylabel="P [MW]",
    plot_title="Storage charging power (combo)",
    filename="combo_storage_charging",
)

pp_plot.plot_timeseries_result(
    data_df=storage_df[["SOC_MWh"]],
    ylabel="Energy [MWh]",
    plot_title="Storage SOC (combo)",
    filename="combo_storage_soc",
)


/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.

